In [1]:
# Document와 텍스트 분할 클래스가 든 패키지를 정해진 버전으로 설치합니다.
# -q는 설치 과정을 짧게 표시해 이후 실행 결과를 읽기 쉽게 합니다.
%pip install -q "langchain==1.4.0" "langchain-text-splitters==1.1.2"


Note: you may need to restart the kernel to use updated packages.


<h3>개념 요약</h3>
<p>로더는 원문과 출처 정보를 <code>Document</code> 객체로 묶습니다. 스플리터는 문서를 검색 단위인 청크로 나누며 메타데이터를 각 청크에 복사합니다.</p>

In [2]:
# version()은 현재 커널에 설치된 패키지 버전 문자열을 반환합니다.
# 두 출력으로 노트북이 의도한 LangChain 환경인지 확인합니다.
from importlib.metadata import version

print('langchain', version('langchain'))
print('langchain-text-splitters', version('langchain-text-splitters'))


langchain 1.4.0
langchain-text-splitters 1.1.2


<h3>예제 실행</h3>
<p>짧은 문장과 출처 메타데이터를 하나의 <code>Document</code> 객체로 만듭니다.</p>

In [3]:
# Document는 page_content에 본문, metadata에 출처 정보를 보관합니다.
# docs는 문서 여러 건을 담는 리스트이며, 예제에서는 D01 한 건만 담습니다.
# docs[0]의 0은 리스트의 첫 번째 문서를 선택하는 인덱스입니다.
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = '파란도서관은 월요일에 쉽니다. 화요일부터 일요일까지 운영합니다.'
docs = [Document(page_content=text, metadata={'doc_id': 'D01', 'page': 1})]
print(docs[0])


page_content='파란도서관은 월요일에 쉽니다. 화요일부터 일요일까지 운영합니다.' metadata={'doc_id': 'D01', 'page': 1}


<h3>분할 결과 확인</h3>
<p>문서를 나눈 뒤에도 각 청크에 출처 정보가 남는지 확인합니다.</p>

In [4]:
# chunk_size는 청크의 최대 길이, chunk_overlap은 이웃 청크가 공유할 길이입니다.
# split_documents()는 Document 리스트를 받아 메타데이터를 복사한 청크 리스트를 반환합니다.
# for는 반환된 청크를 하나씩 chunk에 담아 본문과 메타데이터를 출력합니다.
splitter = RecursiveCharacterTextSplitter(chunk_size=24, chunk_overlap=4)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    print(chunk.page_content, chunk.metadata)


파란도서관은 월요일에 쉽니다. 화요일부터 {'doc_id': 'D01', 'page': 1}
일요일까지 운영합니다. {'doc_id': 'D01', 'page': 1}


<h3>변형 실행</h3>
<p><code>chunk_size</code>를 18로 바꿨을 때 청크 수가 어떻게 달라지는지 확인합니다.</p>

In [5]:
# smaller는 겹침은 유지하고 청크 크기만 18로 줄인 비교용 스플리터입니다.
# len()은 split_documents()가 반환한 small_chunks 리스트의 요소 개수를 반환합니다.
smaller = RecursiveCharacterTextSplitter(chunk_size=18, chunk_overlap=4)
small_chunks = smaller.split_documents(docs)
print('청크 수:', len(small_chunks))


청크 수: 3


<h3>실행 검증</h3>
<p>앞 셀의 결과가 이 절에서 확인할 조건을 만족하는지 단언문으로 검사합니다.</p>

In [6]:
# 첫 assert의 all()은 모든 청크의 doc_id가 D01일 때만 True를 반환합니다.
# 둘째 assert는 청크 크기를 줄인 결과가 기존 결과보다 적지 않은지 검사합니다.
assert chunks and all(c.metadata['doc_id'] == 'D01' for c in chunks)
assert len(small_chunks) >= len(chunks)
print('검증 통과: 내용과 출처 정보가 함께 분할되었습니다.')


검증 통과: 내용과 출처 정보가 함께 분할되었습니다.


<h3>정리</h3>
<p>검색 결과를 출처와 연결하려면 청크를 만들기 전에 문서 ID와 쪽 번호를 기록해야 합니다. 청크 크기와 겹침은 다음 실험에서 바꿀 설정입니다.</p>

<p><strong>원인</strong> — 겹침 길이를 청크 크기보다 크게 지정하면 분할 기준이 성립하지 않습니다.</p>

In [7]:
# 청크 크기 10보다 큰 겹침 11은 유효한 분할 설정이 아닙니다.
# try는 생성자 호출을 시도하고, except는 라이브러리가 발생시킨 ValueError만 error에 담습니다.
try:
    RecursiveCharacterTextSplitter(chunk_size=10, chunk_overlap=11)
except ValueError as error:
    print('예상한 실패:', error)


예상한 실패: Got a larger chunk overlap (11) than chunk size (10), should be smaller.


<p><strong>수정</strong> — <code>chunk_overlap</code>을 <code>chunk_size</code>보다 작게 바꾸고 같은 문서를 다시 나눕니다.</p>

In [8]:
# fixed_splitter는 청크 크기 10과 더 작은 겹침 2를 사용합니다.
# fixed_chunks는 수정한 설정으로 나눈 Document 리스트입니다.
# 리스트가 비지 않고 모든 청크에 D01이 보존되었는지 두 assert로 확인합니다.
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10, chunk_overlap=2
)
fixed_chunks = fixed_splitter.split_documents(docs)
print('After 청크 수:', len(fixed_chunks))
assert fixed_chunks
assert all(c.metadata['doc_id'] == 'D01' for c in fixed_chunks)


After 청크 수: 5
